SETUP

In [0]:
import requests
import json
from datetime import datetime
from pyspark.sql.types import StructType, StructField, StringType, DateType
from config.f1_config import add_ingestion_metadata,CATALOG

API_URL = "https://api.jolpi.ca/ergast/f1/current/drivers.json"
STAGING_PATH = "abfss://staging@stf1project.dfs.core.windows.net/drivers"
BRONZE_TABLE = "f1_workspace.bronze.drivers"

season = datetime.now().year
run_date = datetime.now().strftime("%Y-%m-%d")

API CALL

In [0]:
response = requests.get(API_URL, timeout=30)
response.raise_for_status()

data = response.json()
drivers_list = data["MRData"]["DriverTable"]["Drivers"]

print(f"Pulled {len(drivers_list)} drivers for season {season}")

LAND DATA IN ADLS

In [0]:
raw_path = f"{STAGING_PATH}/season={season}/run_date={run_date}/drivers.json"

dbutils.fs.put(raw_path, json.dumps(drivers_list), overwrite=True)

print(f"Landed raw JSON at: {raw_path}")

EXPLICIT SCHEMA

In [0]:
driver_schema = StructType([
    StructField("dateOfBirth", StringType(), True),
    StructField("driverid", StringType(), True),
    StructField("familyName", StringType(), True),
    StructField("givenName", StringType(), True),
    StructField("name", StringType(), True),
    StructField("nationality", StringType(), True),
    StructField("permanentNumber", StringType(), True),
    StructField("url", StringType(), True),
])

READ DATA

In [0]:
df = spark.read.schema(driver_schema).json(raw_path)
df = add_ingestion_metadata(df)



WRITE DATA

In [0]:
df.write.format("delta")\
    .mode("overwrite")\
    .option("overwriteSchema", True)\
    .saveAsTable(BRONZE_TABLE)
